# V4 KOLF training：扩展图输入 → 全基因 CRISPRi 残差 → 绝对表达

本 notebook 接在 **V4_KOLF_Data.ipynb** 和 **V4_KOLF_graphs.ipynb** 后运行。

训练契约：

```text
x：matched control 的 V4 扩展图基因表达
图：V4 control SEACells 共表达图
target feature：KOLF 全基因 control 的 124 维特征
c：同一个 matched control 的官方匹配全基因表达
y：真实 KOLF CRISPRi perturbed 的官方匹配全基因表达
Δ = y − c
```

模型学习 `pred_delta`，最终输出：

```text
pred_absolute = max(0, c + pred_delta)
```

OOD 靶点直接读取 Graph 生成的 `V4_ood_split.json`，保证 Graph 的 DEG 选择与 Training 的评估切分完全一致。

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
import os
import sys
import json
import random
import datetime
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
from sklearn import metrics
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
import warnings

PROJECT_DIR = r"E:\VSCODE\PrePR-CT\Cell-Type-Specific-Graphs-main"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import utils
from utils import *
from model import GNN, MLP

# =============================================================================
# V4 configuration
# =============================================================================
config = {
    'data_path': 'Data',
    'graphs_path': 'graphs',
    'graphs_data': 'KOLF_V4',
    'graph_input_h5ad': 'KOLF_V4_graph_input.h5ad',
    'full_dataset_h5ad': 'KOLF_V4_full_QC.h5ad',
    'output_gene_order': 'V4_output_gene_order.csv',
    'ood_split': 'V4_ood_split.json',
    'target_coverage': 'V4_VCC_target_coverage.csv',
    'train_degs': 'V4_train_DEGs.csv',
    'save_path_results': 'Results',
    'save_path_models': 'model_checkpoints',
    'multi_pert': True,
    'params': {
        'hidden_channels': 64,
        'weight_decay': 1e-5,
        'in_head': 1,
        'learning_rate': -3,
        'num_epochs': 100,
        'batch_size': 256,
    },
    'deg_relative_weight': 5.0,
    # delta 为主；绝对表达和逐配对 MSE 辅助；top DEG 额外加权。
    'loss_weights': {
        'delta_emd': 1.0,
        'absolute_emd': 0.1,
        'delta_mse': 0.1,
        'deg_delta_mse': 0.5,
    },
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir = os.path.join(PROJECT_DIR, config['data_path'])
results_dir = os.path.join(PROJECT_DIR, config['save_path_results'])
models_dir = os.path.join(PROJECT_DIR, config['save_path_models'])
graph_dir = os.path.join(PROJECT_DIR, config['graphs_path'], config['graphs_data'])
os.makedirs(results_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, 'KOLF_V4_delta_model.pt')
training_metadata_path = os.path.join(results_dir, 'KOLF_V4_delta_training_metadata.json')
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

print('Device:', device)
print('Model path:', model_path)
print('Loss weights:', config['loss_weights'])

In [ ]:
# -----------------------------------------------------------------------------
# V4 data loading：扩展图输入 + 官方全基因输出 + Graph 的固定 OOD split
# -----------------------------------------------------------------------------
def read_csv(name):
    path = os.path.join(data_dir, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'缺少 V4 输入文件：{path}')
    return path

graph_input_path = read_csv(config['graph_input_h5ad'])
full_input_path = read_csv(config['full_dataset_h5ad'])
output_order_path = read_csv(config['output_gene_order'])
ood_split_path = read_csv(config['ood_split'])
target_coverage_path = read_csv(config['target_coverage'])
train_degs_path = read_csv(config['train_degs'])

graph_adata = sc.read(graph_input_path)
full_adata_all = sc.read(full_input_path)
output_order_df = pd.read_csv(output_order_path)
with open(ood_split_path, encoding='utf-8') as handle:
    ood_split_info = json.load(handle)
target_coverage_df = pd.read_csv(target_coverage_path)
train_degs_df = pd.read_csv(train_degs_path)

required_graph_obs = {'cell_type', 'condition', 'SEACell', 'cov_cond'}
missing_graph_obs = sorted(required_graph_obs - set(graph_adata.obs.columns))
if missing_graph_obs:
    raise KeyError(f'graph_input 缺少字段：{missing_graph_obs}；请先运行 V4_KOLF_graphs.ipynb。')
if not graph_adata.obs_names.equals(full_adata_all.obs_names):
    raise ValueError('graph_input 与 full_QC 的细胞顺序不一致。')
if not output_order_df.output_index.is_unique:
    raise ValueError('V4_output_gene_order.csv 的 output_index 不唯一。')
if (np.sort(output_order_df.output_index.values) != np.arange(len(output_order_df))).any():
    raise ValueError('V4_output_gene_order.csv 的 output_index 不连续。')
output_genes = output_order_df.gene_name.astype(str).tolist()
output_gene_to_pos = {g: i for i, g in enumerate(output_genes)}
if not set(output_genes).issubset(set(full_adata_all.var_names.astype(str))):
    missing = sorted(set(output_genes) - set(full_adata_all.var_names.astype(str)))
    raise ValueError(f'输出基因不在 full_QC 中，示例：{missing[:20]}')

testing_genes = [str(g) for g in ood_split_info.get('testing_genes', [])]
training_targets = [str(g) for g in ood_split_info.get('training_targets', [])]
if not testing_genes or not training_targets:
    raise ValueError('V4_ood_split.json 不完整；请重新运行 V4_KOLF_graphs.ipynb。')
condition_counts = graph_adata.obs.loc[
    graph_adata.obs.condition != 'control', 'condition'
].value_counts()
missing_testing = [g for g in testing_genes if g not in condition_counts.index]
if missing_testing:
    raise ValueError(f'OOD split 中的靶点在当前数据中不存在：{missing_testing}')

full_adata = full_adata_all[:, output_genes].copy()
del full_adata_all

# 单张 V4 图。
graph_files = sorted(f for f in os.listdir(graph_dir) if f.endswith('_coexpr_graph.pkl'))
if len(graph_files) != 1:
    raise ValueError(f'{graph_dir} 中应恰好有 1 张 V4 图，实际为 {len(graph_files)} 张。')
graph = torch.load(os.path.join(graph_dir, graph_files[0]), map_location=device)
graph.x = graph.x.float()
graph.pos = graph.pos.long()
if graph.pos.numel() and (int(graph.pos.min()) < 0 or int(graph.pos.max()) >= graph_adata.n_vars):
    raise ValueError('V4 图节点的 graph.pos 超出 graph_input 基因范围。')
cell_type_key = str(graph_adata.obs.cell_type.iloc[0])

# 图节点均值／方差只来自 control。
def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, 'toarray') else np.asarray(matrix)

ctrl_graph = graph_adata[graph_adata.obs.condition == 'control'].copy()
graph_gene_indices = graph.pos.tolist()
ctrl_graph_expr = torch.tensor(
    to_dense(ctrl_graph[:, graph_gene_indices].X)
).float()
graph.x = torch.cat([
    ctrl_graph_expr.mean(dim=0).unsqueeze(1),
    ctrl_graph_expr.var(dim=0).unsqueeze(1),
], dim=1).float()

# V4：124 维扰动特征来自 full control，不再来自 HVG5000。
def build_control_gene_features(feature_adata, feature_dim=124):
    feature_ctrl = feature_adata[feature_adata.obs.condition == 'control'].copy()
    if len(feature_ctrl) < 10:
        raise ValueError('control 细胞不足，无法构建全基因靶点特征。')
    n_pcs = min(50, feature_dim - 4, feature_ctrl.n_obs - 1, feature_ctrl.n_vars - 1)
    sc.pp.pca(feature_ctrl, n_comps=n_pcs, random_state=RANDOM_SEED)
    features = np.zeros((feature_adata.n_vars, feature_dim), dtype=np.float32)
    pcs = np.asarray(feature_ctrl.varm['PCs'], dtype=np.float32)
    features[:, :n_pcs] = (
        (pcs - pcs.mean(axis=0, keepdims=True)) / (pcs.std(axis=0, keepdims=True) + 1e-6)
    )
    X = feature_ctrl.X
    if issparse(X):
        mean_expr = np.asarray(X.mean(axis=0)).ravel()
        mean_square = np.asarray(X.multiply(X).mean(axis=0)).ravel()
        detection = np.asarray((X > 0).sum(axis=0)).ravel() / X.shape[0]
    else:
        X = np.asarray(X)
        mean_expr = X.mean(axis=0)
        mean_square = np.square(X).mean(axis=0)
        detection = (X > 0).mean(axis=0)
    var_expr = np.maximum(mean_square - np.square(mean_expr), 0)
    features[:, -4] = (mean_expr - mean_expr.mean()) / (mean_expr.std() + 1e-6)
    features[:, -3] = (np.log1p(var_expr) - np.log1p(var_expr).mean()) / (np.log1p(var_expr).std() + 1e-6)
    features[:, -2] = detection.astype(np.float32)
    features[:, -1] = 1.0
    return features

# 在完整 full_adata 上构建，再按输出基因保留每个靶点的特征。
full_feature_adata = full_adata.copy()
full_gene_features = build_control_gene_features(full_feature_adata, feature_dim=124)
full_gene_to_index = {g: i for i, g in enumerate(full_adata.var_names.astype(str))}
del full_feature_adata

target_feature_by_gene = {}
for gene in condition_counts.index.astype(str):
    idx = full_gene_to_index.get(gene)
    if idx is not None:
        target_feature_by_gene[gene] = full_gene_features[idx]
missing_feature_genes = sorted(set(condition_counts.index.astype(str)) - set(target_feature_by_gene))

print('Graph input:', graph_adata.shape)
print('Graph nodes in pkl:', int(graph.num_nodes))
print('Full official output:', full_adata.shape)
print('OOD testing genes:', testing_genes)
print('Training targets:', len(training_targets))
print('Target features from: full control genes')
print('Targets missing full-gene features:', len(missing_feature_genes))
if missing_feature_genes:
    print('Missing feature examples:', missing_feature_genes[:20])
for gene in testing_genes:
    feature = target_feature_by_gene.get(gene)
    print(f'{gene}: feature available={feature is not None}, nonzero={feature is not None and bool(np.any(feature != 0))}')

In [ ]:
# -----------------------------------------------------------------------------
# V4 paired samples：扩展图 control 输入 + 全基因 control 基线 + perturbed 监督
# -----------------------------------------------------------------------------
DEG_RELATIVE_WEIGHT = float(config.get('deg_relative_weight', 5.0))
deg_weight_by_target = {}
deg_record_rows = []
for target_gene, group in train_degs_df.groupby('target_gene', sort=True):
    positions = []
    for gene in group.gene_name.astype(str):
        pos = output_gene_to_pos.get(gene)
        if pos is not None:
            positions.append(pos)
    weight = np.zeros(full_adata.n_vars, dtype=np.float32)
    if positions:
        weight[np.array(sorted(set(positions)), dtype=int)] = DEG_RELATIVE_WEIGHT
    deg_weight_by_target[str(target_gene)] = weight
    deg_record_rows.append({
        'target_gene': str(target_gene),
        'n_weighted_degs': int((weight > 0).sum()),
    })

# 与 V4 Graph 相同的 SEACell 分层 control 匹配。
stim_graph = graph_adata[graph_adata.obs.condition != 'control'].copy()
stim_names = stim_graph.obs_names
full_output_stim = full_adata[stim_names, :]
cell_all_Data = []
pair_summary_rows = []
missing_feature_cov_conds = []

for cov_cond in tq.tqdm(stim_graph.obs.cov_cond.unique(), desc='Building V4 paired cells'):
    cell_type, target_gene = cov_cond.rsplit('_', 1)
    ad_stim = stim_graph[stim_graph.obs.cov_cond == cov_cond].copy()
    num_samples = len(ad_stim)
    ad_ctrl = ctrl_graph[ctrl_graph.obs.cell_type == cell_type].copy()
    selected_ctrl_names = balance_subsample(
        np.array(ad_ctrl.obs.index.values.tolist()),
        np.array(ad_ctrl.obs.SEACell.values.tolist()),
        num_samples,
    )

    x_ctrl = to_dense(graph_adata[selected_ctrl_names, :].X).astype(np.float32)
    control_full = to_dense(full_adata[selected_ctrl_names, :].X).astype(np.float32)
    y_full = to_dense(full_output_stim[ad_stim.obs_names, :].X).astype(np.float32)
    if x_ctrl.shape != (num_samples, graph_adata.n_vars):
        raise ValueError(f'扩展图 control 输入形状错误：{cov_cond}')
    expected_full_shape = (num_samples, full_adata.n_vars)
    if control_full.shape != expected_full_shape or y_full.shape != expected_full_shape:
        raise ValueError(f'control/perturbed 全基因形状错误：{cov_cond}')

    feature = target_feature_by_gene.get(target_gene)
    if feature is None:
        target_feature = np.zeros(124, dtype=np.float32)
        missing_feature_cov_conds.append(cov_cond)
    else:
        target_feature = feature
    target_feature_tensor = torch.from_numpy(target_feature).float()
    deg_weight = deg_weight_by_target.get(
        target_gene, np.zeros(full_adata.n_vars, dtype=np.float32)
    )

    for row in range(num_samples):
        cell_all_Data.append(Data(
            x=torch.from_numpy(x_ctrl[row:row + 1]).float(),
            y=torch.from_numpy(y_full[row:row + 1]).float(),
            control_full=torch.from_numpy(control_full[row:row + 1]).float(),
            cell_type=cell_type,
            cov_drug=cov_cond,
            drug=target_gene,
            target_gene=target_gene,
            pert_label=target_feature_tensor.unsqueeze(0),
        ))
    pair_summary_rows.append({
        'cov_cond': cov_cond,
        'target_gene': target_gene,
        'n_pairs': num_samples,
        'n_weighted_degs': int((deg_weight > 0).sum()),
        'target_feature_nonzero': bool(np.any(target_feature != 0)),
        'is_ood': target_gene in set(testing_genes),
    })

pair_summary_df = pd.DataFrame(pair_summary_rows)
pair_summary_path = os.path.join(results_dir, 'V4_training_pairing_summary.csv')
pair_summary_df.to_csv(pair_summary_path, index=False)
print('Total paired cell samples:', len(cell_all_Data))
print('Pairing summary saved to:', pair_summary_path)
print('Covariates missing target feature:', len(missing_feature_cov_conds))
if missing_feature_cov_conds:
    print(missing_feature_cov_conds[:20])

# 使用 Graph 写出的同一 OOD split。
testing_cov_drug = sorted(set(
    graph_adata.obs.loc[
        graph_adata.obs.condition.isin(testing_genes), 'cov_cond'
    ].astype(str)
))
cells_ood = [s for s in cell_all_Data if s.cov_drug in testing_cov_drug]
cells_train = [s for s in cell_all_Data if s.cov_drug not in testing_cov_drug]
print('Cells (train, OOD):', len(cells_train), len(cells_ood))
print('OOD covariates:', testing_cov_drug)
if not cells_train or not cells_ood:
    raise ValueError('训练集或 OOD 测试集为空。')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
train_dataloader = DataLoader(
    cells_train, batch_size=config['params']['batch_size'], shuffle=True
)
print('Top DEG relative weight:', DEG_RELATIVE_WEIGHT)
print(pd.DataFrame(deg_record_rows).head() if deg_record_rows else 'No train DEGs available.')

In [ ]:
# -----------------------------------------------------------------------------
# V4 residual model：扩展图输入 + 全基因线性 delta 输出头
# -----------------------------------------------------------------------------
class DeltaPredictor(nn.Module):
    """全基因残差输出头：最后不加 Sigmoid，因此可正可负。"""
    def __init__(self, input_dim, output_dim, hidden_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        # 初始 pred_delta = 0，即初始预测等于 matched control。
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)

model = GNN(
    total_genes=graph_adata.n_vars,
    num_perts=max(1, len(set(s.drug for s in cells_train))),
    act_fct='Sigmoid',
    hidden_channels=config['params']['hidden_channels'],
    in_head=config['params']['in_head'],
    multi_pert=config['multi_pert'],
).to(device)
predict_input_dim = graph_adata.n_vars + config['params']['hidden_channels'] + 124
model.lin_predict = DeltaPredictor(
    predict_input_dim, full_adata.n_vars, hidden_dim=1024
).to(device)
model = model.float()

print('Input graph genes:', graph_adata.n_vars)
print('Output delta genes:', full_adata.n_vars)
print('Predictor input dim:', predict_input_dim)
print('Delta output activation: Identity (linear)')

# -----------------------------------------------------------------------------
# V4 training：delta EMD + absolute EMD + paired delta MSE + top-DEG weighted MSE
# -----------------------------------------------------------------------------
def make_deg_weight_batch(targets, output_dim, weight_lookup, device):
    if isinstance(targets, str):
        targets = [targets]
    weights = []
    for target in targets:
        weight = weight_lookup.get(str(target), np.zeros(output_dim, dtype=np.float32))
        weights.append(torch.from_numpy(np.asarray(weight, dtype=np.float32)))
    return torch.stack(weights).to(device).float()

def train_v4_delta(model, num_epochs, lr, weight_decay, cell_type_network,
                   train_loader, loss_weights, multi_pert=True):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    model = model.to(device).float()
    for epoch in tq.tqdm(range(num_epochs), leave=False):
        totals = {
            'loss': 0.0,
            'delta_emd': 0.0,
            'absolute_emd': 0.0,
            'delta_mse': 0.0,
            'deg_delta_mse': 0.0,
        }
        model.train()
        for sample in tq.tqdm(train_loader, leave=False):
            sample = sample.to(device)
            cell_type = sample.cell_type
            ctrl = sample.x.float()
            control_full = sample.control_full.float()
            y = sample.y.float()
            true_delta = y - control_full
            deg_weight = make_deg_weight_batch(
                sample.target_gene, full_adata.n_vars,
                deg_weight_by_target, device
            )
            pert_label = sample.pert_label.float() if multi_pert else None
            cell_graphs_x = {
                c: cell_type_network[c].x.to(device).float()
                for c in np.unique(cell_type)
            }
            cell_graphs_pos = {
                c: cell_type_network[c].pos.to(device).long()
                for c in np.unique(cell_type)
            }
            cell_graphs_edges = {
                c: cell_type_network[c].edge_index.to(device)
                for c in np.unique(cell_type)
            }
            pred_delta = model(
                cell_graphs_x, cell_graphs_edges, cell_type,
                cell_graphs_edges.keys(), ctrl, pert_label, cell_graphs_pos
            )
            # 训练时不 clamp，避免负值区域梯度被截断；导出时再 clamp。
            pred_absolute = control_full + pred_delta

            delta_emd = loss_fct(pred_delta, true_delta, sample.cov_drug)
            absolute_emd = loss_fct(pred_absolute, y, sample.cov_drug)
            squared_delta_error = torch.square(pred_delta - true_delta)
            delta_mse = torch.mean(squared_delta_error)
            deg_delta_mse = torch.mean(squared_delta_error * deg_weight)
            loss = (
                loss_weights['delta_emd'] * delta_emd
                + loss_weights['absolute_emd'] * absolute_emd
                + loss_weights['delta_mse'] * delta_mse
                + loss_weights['deg_delta_mse'] * deg_delta_mse
            )
            if not torch.isfinite(loss):
                raise FloatingPointError(f'非有限 loss：{loss.item()}')
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            totals['loss'] += loss.item()
            totals['delta_emd'] += delta_emd.item()
            totals['absolute_emd'] += absolute_emd.item()
            totals['delta_mse'] += delta_mse.item()
            totals['deg_delta_mse'] += deg_delta_mse.item()
        n_batches = len(train_loader)
        print(
            f"Epoch {epoch}, loss: {totals['loss'] / n_batches:.6f}, "
            f"delta_EMD: {totals['delta_emd'] / n_batches:.6f}, "
            f"absolute_EMD: {totals['absolute_emd'] / n_batches:.6f}, "
            f"delta_MSE: {totals['delta_mse'] / n_batches:.6f}, "
            f"deg_MSE: {totals['deg_delta_mse'] / n_batches:.6f}"
        )
    return model

model = train_v4_delta(
    model=model,
    num_epochs=config['params']['num_epochs'],
    lr=10 ** config['params']['learning_rate'],
    weight_decay=config['params']['weight_decay'],
    multi_pert=config['multi_pert'],
    cell_type_network={cell_type_key: graph},
    train_loader=train_dataloader,
    loss_weights=config['loss_weights'],
)
torch.save(model.state_dict(), model_path)
model.load_state_dict(torch.load(model_path, map_location=device))
print('V4 residual model saved to:', model_path)

training_metadata = {
    'version': 'V4',
    'model_type': 'residual_delta',
    'expression_space': 'normalize_total_log1p',
    'graph_input_genes': int(graph_adata.n_vars),
    'graph_nodes_in_pkl': int(graph.num_nodes),
    'output_genes': int(full_adata.n_vars),
    'target_feature_dim': 124,
    'target_feature_source': 'full control genes',
    'predictor_hidden_dim': 1024,
    'output_activation': 'Identity',
    'supervision': 'true_delta = perturbed_full - matched_control_full',
    'prediction': 'pred_absolute = clamp_min(matched_control_full + pred_delta, 0)',
    'loss_weights': config['loss_weights'],
    'deg_relative_weight': DEG_RELATIVE_WEIGHT,
    'testing_genes': testing_genes,
    'n_training_targets': len(training_targets),
    'output_gene_order_file': os.path.join(data_dir, config['output_gene_order']),
    'software': {
        'python': platform.python_version(),
        'scanpy': sc.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'torch': torch.__version__,
    },
}
with open(training_metadata_path, 'w', encoding='utf-8') as handle:
    json.dump(training_metadata, handle, ensure_ascii=False, indent=2)
print('V4 prediction metadata saved to:', training_metadata_path)

In [ ]:
# -----------------------------------------------------------------------------
# V4 OOD evaluation：绝对表达、delta 和图节点覆盖
# -----------------------------------------------------------------------------
degs_dict = {}
for cov_drug in tq.tqdm(sorted(testing_cov_drug), desc='Computing OOD DEGs'):
    ood_cell = cov_drug.rsplit('_', 1)[0]
    adata_cov = full_adata[
        full_adata.obs.cov_cond.isin([cov_drug, ood_cell + '_control']), :
    ].copy()
    sc.tl.rank_genes_groups(
        adata_cov,
        groupby='cov_cond',
        rankby_abs=True,
        method='t-test',
        corr_method='benjamini-hochberg',
        reference=ood_cell + '_control',
        n_genes=len(adata_cov.var),
    )
    dedf_all = sc.get.rank_genes_groups_df(adata_cov, group=cov_drug)
    dedf = dedf_all.loc[dedf_all['pvals_adj'] < 0.05].copy()
    if len(dedf) < 10:
        dedf = dedf_all.copy()
    DEGs_name = rank_genes(dedf)
    degs_dict[cov_drug] = np.where(full_adata.var_names.isin(DEGs_name))[0]

graph_gene_set = set(graph_adata.var_names.astype(str))
ood_loader_all = DataLoader(
    cells_ood, batch_size=config['params']['batch_size'], shuffle=False
)
for data in ood_loader_all.dataset:
    data.x = data.x.float()
    data.y = data.y.float()
    data.control_full = data.control_full.float()
    data.pert_label = data.pert_label.float()

def inference_v4_delta(mean_or_std=True, plot=True):
    model.eval()
    predictions, truths, controls, cov_drugs = [], [], [], []
    with torch.no_grad():
        for sample in tq.tqdm(ood_loader_all, leave=False):
            sample = sample.to(device)
            cell_type = sample.cell_type
            cell_graphs_x = {
                c: graph.x.to(device).float() for c in np.unique(cell_type)
            }
            cell_graphs_pos = {
                c: graph.pos.to(device).long() for c in np.unique(cell_type)
            }
            cell_graphs_edges = {
                c: graph.edge_index.to(device) for c in np.unique(cell_type)
            }
            pred_delta = model(
                cell_graphs_x, cell_graphs_edges, cell_type,
                cell_graphs_edges.keys(), sample.x.float(), sample.pert_label.float(),
                cell_graphs_pos
            )
            pred_absolute = (sample.control_full.float() + pred_delta).clamp(min=0)
            predictions.append(pred_absolute.cpu())
            truths.append(sample.y.float().cpu())
            controls.append(sample.control_full.float().cpu())
            cov_drugs.extend(sample.cov_drug)

    pred = torch.cat(predictions, dim=0).numpy()
    truth = torch.cat(truths, dim=0).numpy()
    control = torch.cat(controls, dim=0).numpy()
    cov_drugs = np.array(cov_drugs)
    summary_rows = []

    for cov_drug in sorted(set(cov_drugs)):
        target_gene = cov_drug.rsplit('_', 1)[1]
        feature = target_feature_by_gene.get(target_gene)
        indices = np.where(cov_drugs == cov_drug)[0]
        y_p = truth[indices]
        pred_p = pred[indices]
        control_p = control[indices]
        true_delta = y_p - control_p
        pred_delta = pred_p - control_p
        stat_name = 'mean' if mean_or_std else 'std'
        if mean_or_std:
            x_stat = y_p.mean(axis=0)
            y_stat = pred_p.mean(axis=0)
        else:
            x_stat = y_p.std(axis=0)
            y_stat = pred_p.std(axis=0)

        DEGs = degs_dict[cov_drug]
        deg_gene_names = full_adata.var_names[DEGs].astype(str)
        n_degs_in_graph = int(sum(g in graph_gene_set for g in deg_gene_names))
        mse = np.mean((y_p - pred_p) ** 2)
        baseline_mse = np.mean((y_p - control_p) ** 2)
        delta_mse = np.mean((true_delta - pred_delta) ** 2)
        r2_all = metrics.r2_score(x_stat, y_stat)
        r2_degs = metrics.r2_score(x_stat[DEGs], y_stat[DEGs]) if len(DEGs) else np.nan
        if mean_or_std:
            true_delta_mean = true_delta.mean(axis=0)
            pred_delta_mean = pred_delta.mean(axis=0)
            r2_delta_all = metrics.r2_score(true_delta_mean, pred_delta_mean)
            r2_delta_degs = (
                metrics.r2_score(true_delta_mean[DEGs], pred_delta_mean[DEGs])
                if len(DEGs) else np.nan
            )
        else:
            r2_delta_all, r2_delta_degs = np.nan, np.nan

        summary_rows.append({
            'cov_cond': cov_drug,
            'target_gene': target_gene,
            'metric': stat_name,
            'target_in_graph': bool(target_gene in graph_gene_set),
            'target_feature_nonzero': bool(feature is not None and np.any(feature != 0)),
            'n_DEGs': int(len(DEGs)),
            'n_DEGs_in_graph': n_degs_in_graph,
            'DEG_graph_coverage': n_degs_in_graph / len(DEGs) if len(DEGs) else np.nan,
            'mse_cell_gene': mse,
            'control_baseline_mse_cell_gene': baseline_mse,
            'delta_mse_cell_gene': delta_mse,
            'R2_all_genes': r2_all,
            'R2_top100_DEGs': r2_degs,
            'R2_delta_all_genes': r2_delta_all,
            'R2_delta_top100_DEGs': r2_delta_degs,
        })
        print(f'{cov_drug} DEG graph coverage: {n_degs_in_graph}/{len(DEGs)}')
        print(f'{cov_drug} target in graph: {target_gene in graph_gene_set}, feature nonzero: {feature is not None and np.any(feature != 0)}')
        print(f'{cov_drug} {stat_name} MSE: {mse:.6f}')
        print(f'{cov_drug} control baseline MSE: {baseline_mse:.6f}')
        print(f'{cov_drug} {stat_name} R2 all genes: {r2_all:.4f}')
        print(f'{cov_drug} {stat_name} R2 top100 DEGs: {r2_degs:.4f}')
        if mean_or_std:
            print(f'{cov_drug} mean delta R2 all genes: {r2_delta_all:.4f}')
            print(f'{cov_drug} mean delta R2 top100 DEGs: {r2_delta_degs:.4f}')

        if plot and len(x_stat) > 1:
            plt.figure(figsize=(6, 6))
            sns.regplot(x=x_stat, y=y_stat, ci=None, color='#1C2E54')
            plt.xlabel(f'True {stat_name} absolute expression')
            plt.ylabel(f'Predicted {stat_name} absolute expression')
            plt.title(cov_drug)
            plt.show()
            if mean_or_std:
                plt.figure(figsize=(6, 6))
                sns.regplot(x=true_delta_mean, y=pred_delta_mean, ci=None, color='#8C3B3B')
                plt.axhline(0, color='grey', linewidth=0.8)
                plt.axvline(0, color='grey', linewidth=0.8)
                plt.xlabel('True mean delta (perturbed - matched control)')
                plt.ylabel('Predicted mean delta')
                plt.title(cov_drug + ' delta')
                plt.show()

    return pd.DataFrame(summary_rows)

print('---------------- Mean absolute expression + delta ----------------')
mean_summary = inference_v4_delta(mean_or_std=True, plot=True)
print(mean_summary)
mean_summary.to_csv(os.path.join(results_dir, 'KOLF_V4_mean_summary.csv'), index=False)

print('---------------- Std absolute expression -------------------------')
std_summary = inference_v4_delta(mean_or_std=False, plot=True)
print(std_summary)
std_summary.to_csv(os.path.join(results_dir, 'KOLF_V4_std_summary.csv'), index=False)

combined_summary = pd.concat([mean_summary, std_summary], ignore_index=True)
combined_path = os.path.join(results_dir, 'KOLF_V4_evaluation_summary.csv')
combined_summary.to_csv(combined_path, index=False)
print('Combined V4 evaluation summary saved to:', combined_path)